## Using Linear Regression to Understand Causality.
05 — The Unreasonable Effectiveness of Linear Regression <br>
06 — Grouped and Dummy Regression <br>
07 — Beyond Confounders

In [9]:
import pandas as pd
import numpy as np
from scipy.special import expit
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib import style
import statsmodels.formula.api as smf
style.use("fivethirtyeight")
import os
os.getcwd()
os.chdir("/Users/hiro/Documents/github/Causal_Inference/")

## Lecture 5: Regression

Observed outcome is: 
$Y_i = Y_{0i} + T_i(Y_{1i}-Y_{0i})$ <br>

Average Treatment Effect is:
$ATE = E[Y_{1}-Y_{0}]$ <br>

If treatment binary and randomized, then kappa is just the difference in means, ATE

$$Y_i = \beta_0 + \kappa T_i + u_i,$$

**Heuristic:** with only a treatment dummy, regression is just a fancy way of comparing treated and control averages.

Example:
$exam_i = \beta_0 + \kappa Online_i + u_i$ <br>


Now for multivariate:

In a multivariate regression
$$
Y_i = \beta_0 + \kappa T_i + \beta'X_i + u_i,$$
the treatment coefficient can be written as

$$\kappa = \frac{\operatorname{Cov}(Y_i,\tilde T_i)}{\operatorname{Var}(\tilde T_i)},$$

where $\tilde T_i$ is the residual from regressing $T_i$ on $X_i$.

**Heuristic:** first remove from treatment everything explained by the controls; then see how the remaining variation in treatment is associated with the outcome.

So regression tries to make treatment **as good as random conditional on \(X\)**.

$y_i = \beta_0 + \kappa T_i + \beta_1 X_{1i} + \dots + \beta_{k}X_{ki} + u_i$ <br>

Now, for non-random data:  regression helps because it compares units with the same observed controls $X$. For example, if we control for IQ, experience, family background, etc., the coefficient on education is the effect of education **holding those factors fixed**.

But this is causal only if the relevant confounders are in \(X\).

---

## Omitted variable bias

If the true model is

$Y_i = \alpha + \kappa T_i + \beta' A_i + u_i$

but we omit $A_i$, then the short regression coefficient is

$$\frac{\operatorname{Cov}(Y_i,T_i)}{\operatorname{Var}(T_i)} =\kappa + \beta'\delta_A,$$

where $\delta_A$ is the effect of the omitted variable on treatment.

**Heuristic:** omitted variable bias appears when the omitted variable affects both:
1. the outcome, and
2. the treatment.

So there is no OVB if the omitted variable affects only one side, or neither.

---

## Core takeaway

Regression is powerful because:

1. in randomized settings, it recovers the treatment effect directly;
2. with controls, it residualizes treatment and compares outcome variation to the part of treatment not explained by \(X\);
3. in observational data, it is causal only under a **no omitted confounders** type assumption.

In short:

> Regression does not magically create causality.  
> It isolates variation in treatment.  
> That variation is causal only if the remaining variation is not confounded.

**How to know if all confounders included?** I’m sorry to bring it to you, but that will depend on our ability to argue in favor or against that fact that all confounders have been included in the model. Personally, I think they haven’t. 

## Chapter 6: Grouped and Dummy Regression

This phenomenon of having a region of low variance and another of high variance is called heteroskedasticity. Grouped data are common. Because of confidentiality. Governments and firms can’t give away personal data because of privacy. If export data, do by means grouping data. Individuals then are no longer uniquely identifiable.

In [43]:
wage = pd.read_csv("./data/wage.csv")[["wage", "lhwage", "educ", "IQ"]]

wage.head()

model_1 = smf.ols('lhwage ~ educ', data=wage).fit()
model_1.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,2.2954,0.089,25.754,0.000,2.121,2.470
educ,0.0529,0.007,8.107,0.000,0.040,0.066


Fear not! Regression doesn’t need big data to work! What we can do is provide weights to our linear regression model. This way, it will consider groups with higher sample size more than the small groups. Notice how I’ve replaced the smf.ols with smf.wls, for weighted least squares. It’s hard to notice, but it will make all the difference.

Notice how the parameter estimate of educ in the grouped model is very close to the one in the ungrouped data (actually, they are the same in this case). Also, even with only 10 data points, we’ve managed to get a statistically significant coefficient. That’s because, although we have fewer points, grouping also lowers the variance by a lot. Also notice how the standard error is a bit smaller and the t statistics is a bit larger. That’s because some information about the variance is lost, so we have to be more conservative. Once we group the data, we don’t know how large the variance is within each group. Compare the results above with what we would have with the non weighted model below.

The parameter estimate is smaller. What is happening here is that the regression is placing equal weight for all points. If we plot the model along the grouped points, we see that the non weighted model is giving more importance to small points in the lower left than it should. As a consequence, the line has a lower slope.

In [46]:
group_wage = (wage
              .assign(count=1)
              .groupby("educ")
              .agg({"lhwage":"mean", "count":"count"})
              .reset_index())

print(group_wage)

model_2 = smf.wls('lhwage ~ educ', data=group_wage, weights=group_wage["count"]).fit()
model_2.summary().tables[1]

model_3 = smf.ols('lhwage ~ educ', data=group_wage).fit()
model_3.summary().tables[1]

sns.scatterplot(x="educ", y = "lhwage", size="count", legend=False, data=group_wage, sizes=(40, 400))
plt.plot(wage["educ"], model_2.predict(wage["educ"]), c="C1", label = "Weighted")
plt.plot(wage["educ"], model_3.predict(wage["educ"]), c="C2", label = "Non Weighted")
plt.xlabel("Years of Education")
plt.ylabel("Log Hourly Wage")
plt.legend();

   educ    lhwage  count
0     9  2.856475     10
1    10  2.786911     35
2    11  2.855997     43
3    12  2.922168    393
4    13  3.021182     85
5    14  3.042352     77
6    15  3.090766     45
7    16  3.176184    150
8    17  3.246566     40
9    18  3.144257     57


## Regression for Dummies

Dummy variables are categorical variables we’ve encoded as binary columns. For example, suppose you have a gender variable that you wish to include in your model. This variable is encoded into 3 categories: male, female and other genders.

## Chapter 7: Beyond Confounders